# multiply-back — ex2: multiply_back with a Python float operand on one side

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `multiply-back`. Running the final beacon cell reports progress against the `Backprop: multiply_back` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: multiply_back` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`multiply-back`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "multiply-back"
DD_SUBTOPIC = "Backprop: multiply_back"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## multiply_back with a Python-float operand — quick refresher

Real call sites mix tensors and scalars: `out = multiply(x, 3.0)`. The forward wrapper stores `3.0` as a raw float in `recipe.args`. The reverse pass then calls `multiply_back0(grad_out, out, x, 3.0)` — `y` is a Python float, NOT a tensor.

Two implementation choices that ex1's tensor-tensor tests do not exercise:
1. **Coerce the float to a 0-D tensor** before the multiplication. `grad_out * 3.0` already works via torch's scalar broadcasting, but `unbroadcast(grad, y)` needs `y.shape` — calling `.shape` on a Python float crashes.
2. **No backward for the float side.** Python scalars don't have `.grad`; `multiply_back1` is still defined for the API uniformity, but the dispatcher only calls it if the corresponding arg was a tracked MiniTensor. So `multiply_back1` is never invoked for the float — but it must not crash if it is.

### Exercise 2 — multiply_back with a Python float operand on one side

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply multiply_back0/back1 to a forward op where one operand is a Python float, coercing as needed and verifying the float-side backward never crashes even though it never accumulates a grad.
> Keywords: multiply-back, scalar, float-operand, coerce, unbroadcast
> ```

**KCs targeted:** `multiply-back`, `arg-position-back-functions`

Implement `multiply_back0(grad_out, out, x, y)` and `multiply_back1(grad_out, out, x, y)` for the forward op `out = multiply(x, y)` where EITHER `x` OR `y` may be a Python float instead of a tensor.

Provided in the cell: `unbroadcast(grad, original)`.

Requirements:

1. **Coerce the partner operand if it's a Python float** so the multiplication produces a tensor. `grad_out * 3.0` works via scalar broadcasting, but `unbroadcast(grad, y)` calls `.shape` on `y` — needs a tensor.
2. **`isinstance(y, Tensor)` test** is the cleanest gate: if `y` is a float, `y = t.tensor(y, dtype=grad_out.dtype)`.
3. **Don't crash on a float in the parent slot.** Even if `y` is a float, `multiply_back1` (the back fn for the y-side) might still be called by a buggy dispatcher — it must return a sane tensor, not raise. We'll test this directly.
4. **Wrap in `unbroadcast(grad, parent)`** for the tensor side.

The drill exercises the mixed-type call: `out = multiply(x, 3.0)`. Verify `multiply_back0(grad_out, out, x, 3.0)` produces `grad_out * 3.0` reshaped to `x`'s shape, and the float-side back fn doesn't crash.

Inputs raw `torch.Tensor` or Python float. No autograd.

In [ ]:
def unbroadcast(grad: Tensor, original: Tensor) -> Tensor:
    while grad.ndim > original.ndim:
        grad = grad.sum(dim=0)
    for i, size in enumerate(original.shape):
        if size == 1 and grad.shape[i] != 1:
            grad = grad.sum(dim=i, keepdim=True)
    return grad


def multiply_back0(grad_out, out, x, y) -> Tensor:
    """dL/dx for out = x * y. y may be a Python float."""
    raise NotImplementedError()


def multiply_back1(grad_out, out, x, y) -> Tensor:
    """dL/dy for out = x * y. x may be a Python float."""
    raise NotImplementedError()


def _test_ex2():
    # --- tensor * float, back through the tensor side ---
    x = t.tensor([2.0, 3.0, 4.0])
    y_f = 5.0
    out = x * y_f
    g0 = multiply_back0(t.ones(3), out, x, y_f)
    assert g0.shape == x.shape, f'g0 shape: {g0.shape}'
    assert t.allclose(g0, t.full((3,), 5.0)), f'g0 (=y_f) wrong: {g0}'

    # --- non-unit grad_out ---
    grad_out = t.tensor([10.0, 100.0, 1000.0])
    g0 = multiply_back0(grad_out, out, x, y_f)
    assert t.allclose(g0, grad_out * 5.0), f'chain g0: {g0}'

    # --- float * tensor (flipped argument order) ---
    x_f = 7.0
    y = t.tensor([1.0, 2.0, 4.0])
    out = x_f * y
    g1 = multiply_back1(t.ones(3), out, x_f, y)
    assert g1.shape == y.shape
    assert t.allclose(g1, t.full((3,), 7.0)), f'g1 (=x_f) wrong: {g1}'

    # --- float-side back fn must NOT crash when called ---
    # In a real dispatcher this back fn is skipped for non-tensor parents,
    # but it must be SAFE to call defensively.
    x = t.tensor([2.0, 3.0, 4.0])
    y_f = 5.0
    out = x * y_f
    try:
        g_float_side = multiply_back1(t.ones(3), out, x, y_f)
        # Whatever it returns, must be a tensor — no AttributeError.
        assert isinstance(g_float_side, t.Tensor)
    except AttributeError as e:
        raise AssertionError(f'float-side back fn crashed: {e}')

    # --- broadcasting still works on the tensor side ---
    x_b = t.tensor([[1.0, 2.0, 3.0, 4.0]])  # (1,4)
    y_b = 2.0                                 # scalar
    out_b = x_b * y_b                         # (1,4)
    g0_b = multiply_back0(t.ones(3, 4), out_b, x_b, y_b)
    # Without unbroadcast we'd get (3,4); with unbroadcast we get (1,4) = x_b.shape.
    assert g0_b.shape == x_b.shape, (
        f'expected unbroadcast to (1,4), got {g0_b.shape} '
        f'(did you forget to call unbroadcast for the tensor side?)'
    )
    # Value: 3 rows of ones * 2 broadcast = 6 per column after unbroadcast.
    assert t.allclose(g0_b, t.full((1, 4), 6.0)), f'broadcast unbroadcast: {g0_b}'

    # --- agreement with torch.autograd on tensor*float ---
    x_ref = t.tensor([2.0, 3.0, 4.0], requires_grad=True)
    (x_ref * 5.0).sum().backward()
    g_ours = multiply_back0(t.ones(3), x_ref.detach() * 5.0, x_ref.detach(), 5.0)
    assert t.allclose(g_ours, x_ref.grad, atol=1e-6), 'disagrees with autograd'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def multiply_back0(grad_out, out, x, y) -> Tensor:
    # y may be a Python float — coerce so unbroadcast(., y) works.
    if not isinstance(y, t.Tensor):
        y = t.tensor(y, dtype=grad_out.dtype)
    return unbroadcast(grad_out * y, x)


def multiply_back1(grad_out, out, x, y) -> Tensor:
    # x may be a Python float — coerce same way.
    if not isinstance(x, t.Tensor):
        x = t.tensor(x, dtype=grad_out.dtype)
    if not isinstance(y, t.Tensor):
        # Float-side back fn called defensively: return a 0-D tensor.
        return t.tensor(0.0, dtype=grad_out.dtype)
    return unbroadcast(grad_out * x, y)
```

**Why coerce the partner instead of skipping unbroadcast.** `grad_out * y_f` already works (torch promotes the float), but the follow-up `unbroadcast(result, y_f)` calls `y_f.shape` — Python floats have no `.shape`. The cheapest fix is to coerce `y_f` to a 0-D tensor; `unbroadcast` then collapses correctly to a scalar.

**Why the float-side back fn must not crash.** In production, the dispatcher skips non-tensor parents (the parents dict only contains tensor-valued parents at tracked argnums). But during development, an over-eager dispatcher might still invoke it; raising `AttributeError` here turns into a confusing crash trace 30 frames deep. Returning a `0.0` scalar is the safe defensive answer — the grad is never accumulated anywhere, so the value doesn't matter.

**Why match `grad_out.dtype` when coercing.** `t.tensor(3.0)` defaults to `float32`. If `grad_out` is `float64`, the silent downcast in the multiplication produces a `float32` grad that later crashes in-place accumulation. Pin the dtype to `grad_out.dtype` and the pipeline stays homogeneous.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()